# 04. Modern models and comparative diagnostics

In [1]:
from pathlib import Path
import re, json, hashlib, time, gc, warnings, itertools, platform
import numpy as np
import pandas as pd
import yaml, joblib
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score, matthews_corrcoef, balanced_accuracy_score,
    brier_score_loss, log_loss, confusion_matrix)
from threadpoolctl import threadpool_limits

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                 if (path / "dataset").is_dir() and (path / "notebooks").is_dir())
OUTPUT_ROOT = REPO_ROOT / "artifacts"  # generated tables, models, predictions and figures
(OUTPUT_ROOT / "Configs").mkdir(parents=True, exist_ok=True)
SHARED_DATASET_ROOT = REPO_ROOT / "dataset"
for folder in ["Processed", "Interim", "Features", "Splits"]:
    (SHARED_DATASET_ROOT / folder).mkdir(parents=True, exist_ok=True)
RUN_ID = "racf_20261011"
TABLES = OUTPUT_ROOT / "Tables" / RUN_ID
MODEL_ROOT = OUTPUT_ROOT / "Models" / RUN_ID / "modern"
PRED_ROOT = OUTPUT_ROOT / "Predictions" / RUN_ID / "modern"
for folder in [TABLES, MODEL_ROOT, PRED_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.precision", 4)
protocol = yaml.safe_load((OUTPUT_ROOT / "Configs/experiment.yaml").read_text())
assert not protocol["test_opened_for_model_selection"]
SEEDS = protocol["seeds"]
N_JOBS = 4
thread_limiter = threadpool_limits(limits=N_JOBS)

import torch
from torch import nn
from torch.nn import functional as F
from sklearn.linear_model import RidgeClassifier
from aeon.transformations.collection.convolution_based import MultiRocket, HydraTransformer
import copy, os
torch.set_num_threads(4)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

import psutil, threading


In [2]:
def scientific_table(frame: pd.DataFrame, name: str, caption: str) -> None:
    """Export and validate all results; display the selected narrative tables."""
    frame.to_csv(TABLES / f"{name}.csv", index=False)
    assert not frame.isna().any().any(), f"Explain undefined statistics in {name}"
    # Category-based curation applies to all models, irrespective of their results.
    if name in {"nb04_hydra_batch_equivalence", "nb04_checkpoint_reproduction", "nb04_selected_solver_convergence"} or name.startswith(
        ("nb04_trials_", "nb04_seeds_", "nb04_regime_",
         "nb04_site_errors_", "nb04_reactor_errors_")):
        return
    display(Markdown(f"**{caption}**"))
    display(frame.style.format(scientific_number).hide(axis="index"))


def scientific_number(value, decimals=4):
    """Typography only: preserve raw numeric values and Python arithmetic."""
    if pd.isna(value):
        raise ValueError("An undefined statistic requires an explicit scientific explanation")
    if isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_)):
        return str(value).replace("-", "−")
    if isinstance(value, (float, np.floating)):
        if np.isinf(value):
            return "∞" if value>0 else "−∞"
        if value == 0:
            return f"{0:.{decimals}f}"
        if abs(value) < 10**(-decimals):
            return f"{value:.2e}".replace("-", "−")
        return f"{value:.{decimals}f}".replace("-", "−")
    return re.sub(r"(?<![A-Za-z0-9])-(?=\d)|(?<=[eE])-(?=\d)", "−", str(value))


## 1. Comparability and model selection

In [3]:
FEATURES_ROOT = SHARED_DATASET_ROOT / "Features"
SPLITS_ROOT = SHARED_DATASET_ROOT / "Splits"
features = pd.read_parquet(FEATURES_ROOT / "features_raw.parquet")
metadata = pd.read_parquet(FEATURES_ROOT / "observation_metadata.parquet")
schema = json.loads((FEATURES_ROOT / "feature_schema.json").read_text())
assert features.columns.tolist() == schema["feature_names"]
splits = {name:{role:np.load(SPLITS_ROOT / f"{name}_{role}.npy") for role in ["train","evaluation"]}
          for name in ["forward_2020","forward_2021","forward_2022","development","site_transfer"]}
assert len(features) == len(metadata) and len(features.columns) == 51
assert metadata.iloc[splits["development"]["train"]].date.max().year == 2022
assert metadata.iloc[splits["development"]["evaluation"]].date.max().year == 2023
transfer_sites = set(metadata.iloc[splits["site_transfer"]["evaluation"]].plant_id)
# The full predeclared assignment also excludes held sites absent from 2023 evaluation.
site_assignment = pd.read_csv(SPLITS_ROOT / "site_holdout_assignment.csv")
sequences = np.load(FEATURES_ROOT / "sequences_raw.npy",mmap_mode="r")
scientific_table(pd.DataFrame([{"Partition":name, "Training n":len(part["train"]),
    "Evaluation n":len(part["evaluation"]), "Positive days":int(metadata.iloc[part["evaluation"]].target.sum()),
    "Evaluation sites":metadata.iloc[part["evaluation"]].plant_id.nunique()}
    for name, part in splits.items()]), "nb04_evaluation_cohorts", "Common evaluation cohorts; positive days are not independent observations")


**Common evaluation cohorts; positive days are not independent observations**

Partition,Training n,Evaluation n,Positive days,Evaluation sites
forward_2020,28287,20555,46,54
forward_2021,57744,19351,36,54
forward_2022,87425,19020,35,51
development,116904,20333,42,50
site_transfer,80198,5634,15,15


In [4]:
# Inspect the predeclared site flag by its fixed schema, without consulting outcomes.
assert site_assignment.columns.tolist() == ["plant_id", "held_out"]
held_sites = set(site_assignment.loc[site_assignment.held_out, "plant_id"])
assert transfer_sites <= held_sites
forward_names = ["forward_2020", "forward_2021", "forward_2022"]
search_splits = {"temporal":{name:splits[name] for name in forward_names},
    "transfer":{name:{role:idx[~metadata.iloc[idx].plant_id.isin(held_sites).to_numpy()]
                      for role,idx in splits[name].items()} for name in forward_names}}
for scope, partitions in search_splits.items():
    for fold, part in partitions.items():
        train_meta, eval_meta = metadata.iloc[part["train"]], metadata.iloc[part["evaluation"]]
        assert (eval_meta.date.min() - train_meta.date.max()).days > 30
        assert set(train_meta.episode_id).isdisjoint(set(eval_meta.episode_id))
        if scope == "transfer":
            assert set(train_meta.plant_id).isdisjoint(held_sites)
            assert set(eval_meta.plant_id).isdisjoint(held_sites)


## 2. Transformations fitted within training folds

In [5]:
class FoldRepresentation:
    """Train-only preprocessing for raw sequence and common context."""
    def fit(self, indices, window):
        self.window = window
        raw = np.asarray(sequences[indices,:,-window:],dtype=np.float32)
        self.sequence_median = np.array([np.nanmedian(raw[:,j]) if np.isfinite(raw[:,j]).any() else 0 for j in range(raw.shape[1])],dtype=np.float32)
        filled = np.where(np.isfinite(raw),raw,self.sequence_median[None,:,None])
        self.sequence_mean = filled.mean(axis=(0,2))
        self.sequence_scale = np.maximum(filled.std(axis=(0,2)),1e-4)
        self.table = Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("scale",StandardScaler())])
        self.table.fit(features.iloc[indices])
        return self
    def transform(self, indices):
        return self.transform_arrays(sequences[indices,:,-self.window:],features.iloc[indices])
    def transform_arrays(self, raw, context_frame):
        raw = np.asarray(raw,dtype=np.float32)
        assert raw.shape[2] == self.window
        observed = np.isfinite(raw)
        filled = np.where(observed,raw,self.sequence_median[None,:,None])
        normalized = np.clip((filled-self.sequence_mean[None,:,None])/self.sequence_scale[None,:,None],-20,20)
        temporal = np.concatenate([normalized,observed.astype(np.float32)],axis=1)
        context = np.clip(self.table.transform(context_frame),-20,20).astype(np.float32)
        assert np.isfinite(temporal).all() and np.isfinite(context).all()
        return temporal,context


## 3. Neural comparator architectures

In [6]:
class CausalResidual(nn.Module):
    def __init__(self, incoming, width, dilation):
        super().__init__()
        self.padding = 2*dilation
        self.first = nn.Conv1d(incoming,width,3,dilation=dilation,padding=self.padding)
        self.second = nn.Conv1d(width,width,3,dilation=dilation,padding=self.padding)
        self.shortcut = nn.Conv1d(incoming,width,1) if incoming!=width else nn.Identity()
        self.dropout = nn.Dropout(.1)
    def forward(self,x):
        h = self.dropout(F.relu(self.first(x)[:,:,:-self.padding]))
        h = self.dropout(F.relu(self.second(h)[:,:,:-self.padding]))
        return F.relu(h+self.shortcut(x))

class InceptionModule(nn.Module):
    def __init__(self,incoming,width):
        super().__init__()
        self.bottleneck = nn.Conv1d(incoming,width,1,bias=False)
        self.convolutions = nn.ModuleList([nn.Conv1d(width,width,k,padding=k//2,bias=False) for k in [3,5,9]])
        self.pool = nn.Sequential(nn.MaxPool1d(3,stride=1,padding=1),nn.Conv1d(incoming,width,1,bias=False))
        self.norm = nn.BatchNorm1d(4*width)
    def forward(self,x):
        b = self.bottleneck(x)
        return F.relu(self.norm(torch.cat([conv(b) for conv in self.convolutions]+[self.pool(x)],dim=1)))

class TemporalComparator(nn.Module):
    def __init__(self,family,channels,context_dim,width):
        super().__init__()
        self.family=family
        if family=="TCN + context":
            self.encoder=nn.Sequential(*[CausalResidual(channels if j==0 else width,width,2**j) for j in range(3)])
            hidden=width
        else:
            self.encoder=nn.ModuleList([InceptionModule(channels if j==0 else 4*width,width) for j in range(6)])
            self.residuals=nn.ModuleList([nn.Sequential(nn.Conv1d(channels,4*width,1,bias=False),nn.BatchNorm1d(4*width)),nn.Sequential(nn.Conv1d(4*width,4*width,1,bias=False),nn.BatchNorm1d(4*width))])
            hidden=4*width
        self.context=nn.Sequential(nn.Linear(context_dim,32),nn.ReLU(),nn.Dropout(.1))
        self.head=nn.Sequential(nn.Linear(hidden+32,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x,z):
        if self.family=="TCN + context":
            h=self.encoder(x)[:,:,-1]
        else:
            h=x; shortcut=x
            for j,block in enumerate(self.encoder):
                h=block(h)
                if (j+1)%3==0:
                    h=F.relu(h+self.residuals[j//3](shortcut)); shortcut=h
            h=h.mean(dim=2)
        return self.head(torch.cat([h,self.context(z)],dim=1)).squeeze(-1)


## 4. Training and prediction retention

In [7]:
MODERN_GRIDS = {
    "MultiRocket + context":[{"window":14,"kernels":252,"alpha":1.0},{"window":30,"kernels":252,"alpha":1.0},{"window":30,"kernels":504,"alpha":10.0}],
    "Hydra + context":[{"window":14,"groups":32,"alpha":1.0},{"window":30,"groups":32,"alpha":1.0},{"window":30,"groups":64,"alpha":10.0}],
    "Inception + context":[{"window":w,"width":8,"epochs":16,"lr":.001} for w in [7,14,30]],
    "TCN + context":[{"window":w,"width":32,"epochs":16,"lr":.001} for w in [7,14,30]]}
FAMILIES = list(MODERN_GRIDS)
model_slugs = {"MultiRocket + context":"multirocket","Hydra + context":"hydra","Inception + context":"inception","TCN + context":"tcn"}
(OUTPUT_ROOT / "Configs/modern_search.json").write_text(json.dumps({"grids":MODERN_GRIDS,"seeds":SEEDS,"selection":"mean forward-fold AP","batch_size":512,"loss":"BCE, positive weight sqrt(n0/n1)","device":str(DEVICE)},indent=2))
for family,configs in MODERN_GRIDS.items():
    scientific_table(pd.DataFrame([{"Configuration":i+1,"Settings":json.dumps(v)} for i,v in enumerate(configs)]),"nb04_plan_"+model_slugs[family],f"{family}: prespecified configurations")
search_rows,prediction_frames,resource_rows,learning_rows,chosen=[],[],[],[],{}


**MultiRocket + context: prespecified configurations**

Configuration,Settings
1,"{""window"": 14, ""kernels"": 252, ""alpha"": 1.0}"
2,"{""window"": 30, ""kernels"": 252, ""alpha"": 1.0}"
3,"{""window"": 30, ""kernels"": 504, ""alpha"": 10.0}"


**Hydra + context: prespecified configurations**

Configuration,Settings
1,"{""window"": 14, ""groups"": 32, ""alpha"": 1.0}"
2,"{""window"": 30, ""groups"": 32, ""alpha"": 1.0}"
3,"{""window"": 30, ""groups"": 64, ""alpha"": 10.0}"


**Inception + context: prespecified configurations**

Configuration,Settings
1,"{""window"": 7, ""width"": 8, ""epochs"": 16, ""lr"": 0.001}"
2,"{""window"": 14, ""width"": 8, ""epochs"": 16, ""lr"": 0.001}"
3,"{""window"": 30, ""width"": 8, ""epochs"": 16, ""lr"": 0.001}"


**TCN + context: prespecified configurations**

Configuration,Settings
1,"{""window"": 7, ""width"": 32, ""epochs"": 16, ""lr"": 0.001}"
2,"{""window"": 14, ""width"": 32, ""epochs"": 16, ""lr"": 0.001}"
3,"{""window"": 30, ""width"": 32, ""epochs"": 16, ""lr"": 0.001}"


In [8]:
def make_predictions(indices, raw, family, seed, fold, scope, config_id):
    keep = ["observation_id","reactor_id","plant_id","date","episode_id","event_episode_id",
            "event_id","target","control_kind","label_source","scram_type"]
    out = metadata.iloc[indices][keep].copy()
    out["row_index"], out["raw_score"] = indices, np.asarray(raw,float)
    out["model"],out["seed"],out["fold"],out["scope"],out["configuration"] = family,seed,fold,scope,config_id
    assert out.observation_id.is_unique and np.isfinite(out.raw_score).all()
    return out.reset_index(drop=True)




In [9]:
def neural_scores(model, temporal, context, batch=1024):
    model.eval(); values=[]
    with torch.inference_mode():
        for start in range(0,len(context),batch):
            x=torch.from_numpy(temporal[start:start+batch]).to(DEVICE)
            z=torch.from_numpy(context[start:start+batch]).to(DEVICE)
            values.append(torch.sigmoid(model(x,z)).cpu().numpy())
    return np.concatenate(values)

def fit_neural(family, config, seed, xt, zt, y, run_label):
    torch.manual_seed(seed); rng=np.random.default_rng(seed)
    model=TemporalComparator(family,xt.shape[1],zt.shape[1],config["width"]).to(DEVICE)
    with torch.no_grad():
        model.head[-1].bias.fill_(float(np.log(y.mean()/(1-y.mean()))))
    optimizer=torch.optim.AdamW(model.parameters(),lr=config["lr"],weight_decay=1e-3)
    criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(np.sqrt((len(y)-y.sum())/y.sum()),device=DEVICE,dtype=torch.float32))
    for epoch in range(config["epochs"]):
        model.train(); order=rng.permutation(len(y)); total=0.0
        for start in range(0,len(y),512):
            batch=order[start:start+512]
            x=torch.from_numpy(xt[batch]).to(DEVICE);z=torch.from_numpy(zt[batch]).to(DEVICE)
            target=torch.from_numpy(y[batch].astype(np.float32)).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss=criterion(model(x,z),target)
            assert torch.isfinite(loss), run_label
            loss.backward(); norm=nn.utils.clip_grad_norm_(model.parameters(),5.0)
            assert torch.isfinite(norm), run_label
            optimizer.step(); total+=float(loss.detach().cpu())*len(batch)
        learning_rows.append({"Run":run_label,"Epoch":epoch+1,"Training BCE":total/len(y)})
    return model


In [10]:
class SparseHydraScaler:
    """Hydra sparse scaling: sqrt counts and zero-frequency regularization."""
    def fit(self,x):
        z=np.sqrt(np.maximum(x,0))
        self.mean=z.mean(axis=0)
        self.scale=z.std(axis=0,ddof=1)+(z==0).mean(axis=0)**4+1e-8
        return self
    def transform(self,x):
        z=np.sqrt(np.maximum(x,0))
        return (z-self.mean)*(z!=0)/self.scale
    def fit_transform(self,x):
        return self.fit(x).transform(x)

class PeakMemory:
    """Sample process resident memory every 50 ms; not total unified GPU memory."""
    def __enter__(self):
        self.process=psutil.Process(); self.peak=self.process.memory_info().rss
        self.stop=threading.Event()
        def sample_loop():
            while not self.stop.wait(.05): self.sample()
        self.thread=threading.Thread(target=sample_loop,daemon=True);self.thread.start()
        return self
    def sample(self):
        self.peak=max(self.peak,self.process.memory_info().rss)
    def __exit__(self,*exception):
        self.sample();self.stop.set();self.thread.join()

def convolution_features(transform, values, batch=512):
    """Bound Hydra temporary tensors; filters and per-row calculations are unchanged."""
    if isinstance(transform,HydraTransformer):
        return np.concatenate([transform.transform(values[start:start+batch]) for start in range(0,len(values),batch)],axis=0)
    return transform.transform(values)

probe_indices=splits["forward_2020"]["train"][:512]
probe_x,_=FoldRepresentation().fit(probe_indices,14).transform(probe_indices[:64])
probe_transform=HydraTransformer(n_kernels=8,n_groups=32,n_jobs=N_JOBS,random_state=42,output_type="numpy").fit(probe_x)
probe_full=probe_transform.transform(probe_x)
probe_batches=convolution_features(probe_transform,probe_x,batch=16)
probe_error=float(np.max(np.abs(probe_full-probe_batches)))
assert np.allclose(probe_full,probe_batches,rtol=1e-6,atol=1e-5)
scientific_table(pd.DataFrame([{"Real training windows":len(probe_x),"Batch size":16,"Maximum absolute difference":probe_error}]),"nb04_hydra_batch_equivalence","Batched Hydra equivalence on real training windows")
del probe_x,probe_transform,probe_full,probe_batches


In [11]:
def fit_and_predict(family, config_id, seed, scope, fold, partition, save_model=False):
    with PeakMemory() as memory:
        config=MODERN_GRIDS[family][config_id-1]
        tr,ev=partition["train"],partition["evaluation"]
        assert metadata.iloc[tr].date.max()<metadata.iloc[ev].date.min()
        y=metadata.iloc[tr].target.to_numpy(int)
        t0=time.perf_counter()
        representation=FoldRepresentation().fit(tr,config["window"])
        xt,zt=representation.transform(tr)
        if family in ["MultiRocket + context","Hydra + context"]:
            transform=(MultiRocket(n_kernels=config["kernels"],n_jobs=N_JOBS,random_state=seed) if family=="MultiRocket + context" else HydraTransformer(n_kernels=8,n_groups=config["groups"],n_jobs=N_JOBS,random_state=seed,output_type="numpy"))
            transform.fit(xt)
            ft=convolution_features(transform,xt).astype(np.float32)
            scale=StandardScaler() if family=="MultiRocket + context" else SparseHydraScaler()
            ft=scale.fit_transform(ft).astype(np.float32)
            model=RidgeClassifier(alpha=config["alpha"],class_weight="balanced",solver="lsqr",tol=1e-5,max_iter=2000)
            model.fit(np.concatenate([ft,zt],axis=1),y)
            parameter_count=int(model.coef_.size+model.intercept_.size)
            package={"representation":representation,"transform":transform,"scale":scale,"classifier":model,"configuration":config,"family":family}
            fit_seconds=time.perf_counter()-t0
            t1=time.perf_counter(); xv,zv=representation.transform(ev)
            raw=1/(1+np.exp(-np.clip(model.decision_function(np.concatenate([scale.transform(convolution_features(transform,xv)),zv],axis=1)),-25,25)))
            del ft
        else:
            run_label=f"{model_slugs[family]}_{scope}_{fold}_c{config_id}_s{seed}"
            model=fit_neural(family,config,seed,xt,zt,y,run_label)
            parameter_count=sum(p.numel() for p in model.parameters())
            fit_seconds=time.perf_counter()-t0
            t1=time.perf_counter();xv,zv=representation.transform(ev);raw=neural_scores(model,xv,zv)
            package={"representation":representation,"configuration":config,"family":family,"channels":xt.shape[1],"context_dim":zt.shape[1],"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()}}
        predict_seconds=time.perf_counter()-t1
        pred=make_predictions(ev,raw,family,seed,fold,scope,config_id)
        record={"model":family,"seed":seed,"scope":scope,"fold":fold,"configuration":config_id,"training_n":len(tr),"evaluation_n":len(ev),"fit_seconds":fit_seconds,"batch_predict_seconds":predict_seconds,"parameters":parameter_count}
        if save_model:
            checkpoint=MODEL_ROOT / f"{model_slugs[family]}_{scope}_seed{seed}.joblib"
            joblib.dump(package,checkpoint,compress=3);record["model_bytes"]=checkpoint.stat().st_size
        memory.sample();record["Peak process RSS MiB"]=memory.peak/2**20
        record.setdefault("model_bytes",0)  # zero bytes written for unsaved search fits
        resource_rows.append(record)
        pd.DataFrame(resource_rows).to_csv(TABLES / "nb04_fit_resources.csv",index=False)
        pd.DataFrame(learning_rows).to_csv(TABLES / "nb04_learning_curves.csv",index=False)
        del xt,zt,xv,zv,model,package;gc.collect()
        if DEVICE.type=="mps":torch.mps.empty_cache()
        return pred


In [12]:
def search_family(family, scope):
    for config_id, _ in enumerate(MODERN_GRIDS[family], start=1):
        for fold, partition in search_splits[scope].items():
            pred = fit_and_predict(family,config_id,42,scope,fold,partition)
            prediction_frames.append(pred)
            search_rows.append({"Model":family,"Scope":scope,"Configuration":config_id,"Fold":fold,
                "AP":average_precision_score(pred.target,pred.raw_score),
                "ROC-AUC":roc_auc_score(pred.target,pred.raw_score),
                "Brier (raw)":brier_score_loss(pred.target,pred.raw_score)})
        # Persist each completed candidate: a later interruption does not erase its evidence.
        pd.concat([p for p in prediction_frames if p.model.iloc[0]==family and p.scope.iloc[0]==scope]).to_parquet(
            PRED_ROOT / f"search_{model_slugs[family]}_{scope}.parquet",index=False)
        pd.DataFrame(search_rows).to_csv(TABLES / "nb04_search_trials.csv",index=False)
        pd.DataFrame(resource_rows).to_csv(TABLES / "nb04_fit_resources.csv",index=False)
    subset = pd.DataFrame(search_rows).query("Model == @family and Scope == @scope")
    summary = subset.groupby("Configuration",as_index=False).agg(
        **{"Mean AP":("AP","mean"),"Fold SD AP":("AP","std"),"Minimum AP":("AP","min"),
           "Mean ROC-AUC":("ROC-AUC","mean")})
    best = int(summary.sort_values(["Mean AP","Configuration"],ascending=[False,True]).iloc[0].Configuration)
    chosen[(family,scope)] = best
    scientific_table(subset, f"nb04_trials_{model_slugs[family]}_{scope}",
                     f"{family}: all forward-fold evaluations, {scope}")
    scientific_table(summary, f"nb04_selection_{model_slugs[family]}_{scope}",
                     f"{family}: mean across folds; selected configuration {best}, {scope}")
    return best


## 5. Temporal configuration search

### MultiRocket + context

In [13]:
selected_config = search_family("MultiRocket + context", "temporal")
gc.collect();

**MultiRocket + context: mean across folds; selected configuration 3, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.7604,0.0354,0.7241,0.9789
2,0.6824,0.1138,0.5790,0.9826
3,0.7927,0.0432,0.7435,0.9878


### Hydra + context: temporal

In [14]:
selected_config = search_family("Hydra + context", "temporal")
gc.collect();

**Hydra + context: mean across folds; selected configuration 3, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.6816,0.0517,0.6328,0.9966
2,0.6888,0.0690,0.6278,0.9923
3,0.6975,0.1332,0.5540,0.9934


### Inception + context

In [15]:
selected_config = search_family("Inception + context", "temporal")
gc.collect();

**Inception + context: mean across folds; selected configuration 2, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8050,0.0964,0.6958,0.9880
2,0.8296,0.0480,0.7744,0.9906
3,0.8118,0.0201,0.7959,0.9875


### TCN + context

In [16]:
selected_config = search_family("TCN + context", "temporal")
gc.collect();

**TCN + context: mean across folds; selected configuration 3, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.7400,0.0278,0.7146,0.9959
2,0.7747,0.0059,0.7700,0.9964
3,0.8172,0.0307,0.7859,0.9942


## 6. Configuration search excluding held sites

### MultiRocket + context: transfer

In [17]:
selected_config = search_family("MultiRocket + context", "transfer")
gc.collect();

**MultiRocket + context: mean across folds; selected configuration 1, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8616,0.0578,0.8029,0.9927
2,0.8035,0.0083,0.7945,0.9944
3,0.8148,0.0727,0.7509,0.9855


### Hydra + context: transfer

In [18]:
selected_config = search_family("Hydra + context", "transfer")
gc.collect();

**Hydra + context: mean across folds; selected configuration 3, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.7911,0.0241,0.7634,0.9962
2,0.7855,0.0575,0.7514,0.9972
3,0.8231,0.0821,0.7659,0.9932


### Inception + context: transfer

In [19]:
selected_config = search_family("Inception + context", "transfer")
gc.collect();

**Inception + context: mean across folds; selected configuration 3, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8626,0.0550,0.8015,0.9844
2,0.8586,0.0369,0.8290,0.9973
3,0.8871,0.0444,0.8414,0.9991


### TCN + context: transfer

In [20]:
selected_config = search_family("TCN + context", "transfer")
gc.collect();

**TCN + context: mean across folds; selected configuration 3, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8121,0.0195,0.8003,0.9941
2,0.8294,0.0488,0.7924,0.9982
3,0.8518,0.0688,0.7957,0.9987


## 7. Stability across three seeds

In [21]:
selected_oof_frames, development_frames = [], []


def repeat_selected(family, scope):
    config_id = chosen[(family,scope)]
    final_part = splits["development" if scope == "temporal" else "site_transfer"]
    for seed in SEEDS:
        if seed == 42:
            existing = [p for p in prediction_frames if p.model.iloc[0] == family
                and p.scope.iloc[0] == scope and p.configuration.iloc[0] == config_id]
            assert len(existing) == len(forward_names)
            selected_oof_frames.extend(existing)
        else:
            for fold, part in search_splits[scope].items():
                selected_oof_frames.append(fit_and_predict(family,config_id,seed,scope,fold,part))
        pred = fit_and_predict(family,config_id,seed,scope,"development",final_part,save_model=True)
        development_frames.append(pred)
        pd.concat([p for p in selected_oof_frames if p.model.iloc[0]==family and p.scope.iloc[0]==scope]).to_parquet(
            PRED_ROOT / f"oof_{model_slugs[family]}_{scope}.parquet",index=False)
        pd.concat([p for p in development_frames if p.model.iloc[0]==family and p.scope.iloc[0]==scope]).to_parquet(
            PRED_ROOT / f"development_{model_slugs[family]}_{scope}.parquet",index=False)
        pd.DataFrame(resource_rows).to_csv(TABLES / "nb04_fit_resources.csv",index=False)
    rows = [{"Seed":p.seed.iloc[0],"AP":average_precision_score(p.target,p.raw_score),
        "ROC-AUC":roc_auc_score(p.target,p.raw_score),"Positive days":int(p.target.sum())}
        for p in development_frames if p.model.iloc[0]==family and p.scope.iloc[0]==scope]
    scientific_table(pd.DataFrame(rows), f"nb04_seeds_{model_slugs[family]}_{scope}",
                     f"{family}: development 2023, {scope}; parameters selected using preceding folds only")


### MultiRocket + context: temporal

In [22]:
repeat_selected("MultiRocket + context", "temporal")
gc.collect();

### Hydra + context: temporal

In [23]:
repeat_selected("Hydra + context", "temporal")
gc.collect();

### Inception + context: temporal

In [24]:
repeat_selected("Inception + context", "temporal")
gc.collect();

### TCN + context: temporal

In [25]:
repeat_selected("TCN + context", "temporal")
gc.collect();

### MultiRocket + context: transfer

In [26]:
repeat_selected("MultiRocket + context", "transfer")
gc.collect();

### Hydra + context: transfer

In [27]:
repeat_selected("Hydra + context", "transfer")
gc.collect();

### Inception + context: transfer

In [28]:
repeat_selected("Inception + context", "transfer")
gc.collect();

### TCN + context: transfer

In [29]:
repeat_selected("TCN + context", "transfer")
gc.collect();

## 8. TabM as an additional modern tabular comparator

In [30]:
import tabm
MODERN_GRIDS["TabM"]=[{"window":30,"width":64,"dropout":.1,"k":32,"epochs":16,"lr":.001},
    {"window":30,"width":128,"dropout":.1,"k":32,"epochs":16,"lr":.001},
    {"window":30,"width":256,"dropout":.2,"k":32,"epochs":16,"lr":.001}]
FAMILIES.append("TabM"); model_slugs["TabM"]="tabm"
expansion={"family":"TabM","source":"https://github.com/yandex-research/tabm","package_version":tabm.__version__,
    "configurations":MODERN_GRIDS["TabM"],"selection":"mean temporal AP, separate held-site exclusion","test_consulted":False}
(OUTPUT_ROOT / "Configs/nb04_tabm_expansion.json").write_text(json.dumps(expansion,indent=2))

def tabm_network(config,context_dim):
    return tabm.TabM.make(n_num_features=context_dim,d_out=1,n_blocks=3,d_block=config["width"],dropout=config["dropout"],k=config["k"]).to(DEVICE)

def tabm_scores(model,context,batch=1024):
    model.eval();scores=[]
    with torch.inference_mode():
        for start in range(0,len(context),batch):
            logits=model(torch.from_numpy(context[start:start+batch]).to(DEVICE)).squeeze(-1)
            scores.append(torch.sigmoid(logits).mean(1).cpu().numpy())
    return np.concatenate(scores)

fit_reference_and_predict=fit_and_predict

def fit_and_predict(family,config_id,seed,scope,fold,partition,save_model=False):
    if family!="TabM":return fit_reference_and_predict(family,config_id,seed,scope,fold,partition,save_model)
    with PeakMemory() as memory:
        config=MODERN_GRIDS[family][config_id-1];tr,ev=partition["train"],partition["evaluation"]
        assert metadata.iloc[tr].date.max()<metadata.iloc[ev].date.min()
        y=metadata.iloc[tr].target.to_numpy(int);torch.manual_seed(seed);rng=np.random.default_rng(seed)
        t0=time.perf_counter();representation=FoldRepresentation().fit(tr,config["window"])
        zt=np.clip(representation.table.transform(features.iloc[tr]),-20,20).astype(np.float32)
        model=tabm_network(config,zt.shape[1])
        with torch.no_grad():model.output.bias.fill_(float(np.log(y.mean()/(1-y.mean()))))
        optimizer=torch.optim.AdamW(model.parameters(),lr=config["lr"],weight_decay=1e-3)
        criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(np.sqrt((len(y)-y.sum())/y.sum()),device=DEVICE,dtype=torch.float32))
        run_label=f"tabm_{scope}_{fold}_c{config_id}_s{seed}"
        for epoch in range(config["epochs"]):
            model.train();order=rng.permutation(len(y));total=0.0
            for start in range(0,len(y),512):
                rows=order[start:start+512];xb=torch.from_numpy(zt[rows]).to(DEVICE)
                target=torch.from_numpy(y[rows].astype(np.float32)).to(DEVICE)
                optimizer.zero_grad(set_to_none=True);logits=model(xb).squeeze(-1)
                loss=criterion(logits,target[:,None].expand_as(logits));assert torch.isfinite(loss),run_label
                loss.backward();gradient=nn.utils.clip_grad_norm_(model.parameters(),5.0);assert torch.isfinite(gradient),run_label
                optimizer.step();total+=float(loss.detach().cpu())*len(rows)
            learning_rows.append({"Run":run_label,"Epoch":epoch+1,"Training BCE":total/len(y)})
        fit_seconds=time.perf_counter()-t0
        t1=time.perf_counter();zv=np.clip(representation.table.transform(features.iloc[ev]),-20,20).astype(np.float32)
        raw=tabm_scores(model,zv);predict_seconds=time.perf_counter()-t1
        pred=make_predictions(ev,raw,family,seed,fold,scope,config_id)
        record={"model":family,"seed":seed,"scope":scope,"fold":fold,"configuration":config_id,"training_n":len(tr),"evaluation_n":len(ev),
            "fit_seconds":fit_seconds,"batch_predict_seconds":predict_seconds,"parameters":sum(p.numel() for p in model.parameters())}
        if save_model:
            package={"representation":representation,"configuration":config,"family":family,"context_dim":zt.shape[1],"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()}}
            checkpoint=MODEL_ROOT / f"tabm_{scope}_seed{seed}.joblib"
            joblib.dump(package,checkpoint,compress=3);record["model_bytes"]=checkpoint.stat().st_size
        record.setdefault("model_bytes",0);memory.sample();record["Peak process RSS MiB"]=memory.peak/2**20
        resource_rows.append(record);pd.DataFrame(resource_rows).to_csv(TABLES / "nb04_fit_resources.csv",index=False)
        pd.DataFrame(learning_rows).to_csv(TABLES / "nb04_learning_curves.csv",index=False)
        del model,zt,zv;gc.collect()
        if DEVICE.type=="mps":torch.mps.empty_cache()
        return pred

probe_tabm=tabm_network(MODERN_GRIDS["TabM"][0],51)
with torch.inference_mode():
    probe_rows=np.clip(SimpleImputer(keep_empty_features=True).fit_transform(features.iloc[splits["forward_2020"]["train"][:64]]),-20,20).astype(np.float32)
    probe_logits=probe_tabm(torch.from_numpy(probe_rows).to(DEVICE))
assert probe_logits.shape==(64,32,1) and torch.isfinite(probe_logits).all()
scientific_table(pd.DataFrame(expansion["configurations"]),"nb04_tabm_configurations","Registered TabM extension: common budget and verified output shape 64 × 32 × 1")
del probe_tabm,probe_logits,probe_rows;gc.collect();


**Registered TabM extension: common budget and verified output shape 64 × 32 × 1**

window,width,dropout,k,epochs,lr
30,64,0.1000,32,16,0.0010
30,128,0.1000,32,16,0.0010
30,256,0.2000,32,16,0.0010


### TabM configuration search: temporal

In [31]:
selected_config = search_family("TabM", "temporal")
gc.collect();

**TabM: mean across folds; selected configuration 3, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8129,0.0604,0.7436,0.9962
2,0.8273,0.0643,0.7535,0.9972
3,0.8285,0.0606,0.7591,0.9971


### TabM configuration search: transfer

In [32]:
selected_config = search_family("TabM", "transfer")
gc.collect();

**TabM: mean across folds; selected configuration 2, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8590,0.0626,0.7912,0.9982
2,0.8611,0.0661,0.7854,0.9982
3,0.8606,0.0651,0.7878,0.9983


### TabM across three seeds: temporal

In [33]:
repeat_selected("TabM", "temporal")
gc.collect();

### TabM across three seeds: transfer

In [34]:
repeat_selected("TabM", "transfer")
gc.collect();

## 9. OOF calibration and operating points

In [35]:
oof = pd.concat(selected_oof_frames,ignore_index=True)
development = pd.concat(development_frames,ignore_index=True)
assert not oof.duplicated(["model","scope","seed","observation_id"]).any()
assert not development.duplicated(["model","scope","seed","observation_id"]).any()


def raw_logit(values):
    p = np.clip(np.asarray(values,float),1e-8,1-1e-8)
    return np.log(p/(1-p)).reshape(-1,1)


def fit_sigmoid(frame):
    calibrator = LogisticRegression(C=100,solver="lbfgs",max_iter=1000)
    design = np.zeros((len(frame),1)) if frame.model.iloc[0] == "Majority" else raw_logit(frame.raw_score)
    calibrator.fit(design,frame.target.astype(int))
    assert calibrator.coef_[0,0] >= -1e-10
    return calibrator


def select_thresholds(y, score):
    y,score = np.asarray(y,int),np.asarray(score,float)
    candidates = np.unique(np.r_[score,np.nextafter(score.max(),np.inf)])
    order = np.argsort(-score,kind="stable")
    ordered_y, ordered_s = y[order],score[order]
    cum_tp, cum_fp = np.cumsum(ordered_y),np.cumsum(1-ordered_y)
    end = np.r_[ordered_s[:-1] != ordered_s[1:],True]
    thresholds = np.r_[np.nextafter(score.max(),np.inf),ordered_s[end]]
    tp,fp = np.r_[0,cum_tp[end]],np.r_[0,cum_fp[end]]
    fn = y.sum()-tp
    f1 = np.divide(2*tp,2*tp+fp+fn,out=np.zeros(len(tp),float),where=(2*tp+fp+fn)>0)
    best = np.flatnonzero(np.isclose(f1,f1.max(),rtol=0,atol=1e-14))
    threshold_f1 = float(thresholds[best].max())
    allowed = fp/max((y==0).sum(),1)*1000 <= protocol["false_alarm_limit_per_1000_control_days"]
    best_tp = tp[allowed].max()
    threshold_budget = float(thresholds[allowed & (tp==best_tp)].max())
    return threshold_f1,threshold_budget


calibration_rows, threshold_rows = [], []
for (family,scope,seed), frame in oof.groupby(["model","scope","seed"],sort=False):
    calibrator = fit_sigmoid(frame)
    target = (development.model==family)&(development.scope==scope)&(development.seed==seed)
    subset = development.loc[target]
    development.loc[target,"probability"] = calibrator.predict_proba(raw_logit(subset.raw_score))[:,1]
    t_f1,t_budget = select_thresholds(frame.target,frame.raw_score)
    if family == "Majority":
        t_f1,t_budget = np.inf,np.inf
    development.loc[target,"decision_f1"] = subset.raw_score.to_numpy()>=t_f1
    development.loc[target,"decision_budget"] = subset.raw_score.to_numpy()>=t_budget
    threshold_rows.append({"Model":family,"Scope":scope,"Seed":seed,"OOF F1 threshold":t_f1,
        "OOF alarm-budget threshold":t_budget,"OOF positive days":int(frame.target.sum())})
    joblib.dump({"calibrator":calibrator,"threshold_f1":t_f1,"threshold_budget":t_budget,
        "fit_period":"forward OOF 2020-2022","purpose":"development only"},
        MODEL_ROOT / f"calibration_{model_slugs[family]}_{scope}_seed{seed}.joblib",compress=3)
    early, late = frame.query("fold != 'forward_2022'"), frame.query("fold == 'forward_2022'")
    past_calibrator = fit_sigmoid(early)
    late_probability = past_calibrator.predict_proba(raw_logit(late.raw_score))[:,1]
    calibration_rows.append({"Model":family,"Scope":scope,"Seed":seed,
        "2022 Brier, raw":brier_score_loss(late.target,late.raw_score),
        "2022 Brier, sigmoid":brier_score_loss(late.target,late_probability),
        "2022 log loss, sigmoid":log_loss(late.target,late_probability,labels=[0,1])})
development[["decision_f1","decision_budget"]] = development[["decision_f1","decision_budget"]].astype(bool)
oof.to_parquet(PRED_ROOT / "selected_oof.parquet",index=False)
development.to_parquet(PRED_ROOT / "development_predictions.parquet",index=False)
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(threshold_rows).query("Scope == @scope and Seed == 42").drop(columns=["Scope","Seed"]),
        f"nb04_thresholds_{scope}",f"OOF-derived operating points: {scope}; seed 42")
    scientific_table(pd.DataFrame(calibration_rows).query("Scope == @scope and Seed == 42").drop(columns=["Scope","Seed"]),
        f"nb04_calibration_diagnostic_{scope}",f"Transfer of OOF calibration to 2022: {scope}; seed 42")


**OOF-derived operating points: temporal; seed 42**

Model,OOF F1 threshold,OOF alarm-budget threshold,OOF positive days
MultiRocket + context,0.5586,0.5586,117
Hydra + context,0.6302,0.4736,117
Inception + context,0.6478,0.2281,117
TCN + context,0.7227,0.2043,117
TabM,0.8594,0.2469,117


**Transfer of OOF calibration to 2022: temporal; seed 42**

Model,"2022 Brier, raw","2022 Brier, sigmoid","2022 log loss, sigmoid"
MultiRocket + context,0.0758,0.0007,0.0032
Hydra + context,0.0765,0.0009,0.0040
Inception + context,0.0011,0.0008,0.0036
TCN + context,0.0007,0.0006,0.0032
TabM,0.0008,0.0006,0.0027


**OOF-derived operating points: transfer; seed 42**

Model,OOF F1 threshold,OOF alarm-budget threshold,OOF positive days
MultiRocket + context,0.5887,0.5237,80
Hydra + context,0.6656,0.4620,80
Inception + context,0.9574,0.4897,80
TCN + context,0.6925,0.3602,80
TabM,0.9272,0.7955,80


**Transfer of OOF calibration to 2022: transfer; seed 42**

Model,"2022 Brier, raw","2022 Brier, sigmoid","2022 log loss, sigmoid"
MultiRocket + context,0.0740,0.0005,0.0021
Hydra + context,0.0746,0.0006,0.0024
Inception + context,0.0004,0.0005,0.0019
TCN + context,0.0004,0.0004,0.0018
TabM,0.0006,0.0006,0.0023


## 10. Results on the common development cohort

In [36]:
def evaluate_frame(frame, decision="decision_f1"):
    y = frame.target.to_numpy(dtype=int)
    p = frame.probability.to_numpy(float)
    predicted = frame[decision].to_numpy(dtype=bool)
    tn,fp,fn,tp = confusion_matrix(y,predicted,labels=[0,1]).ravel()
    positive = frame.loc[frame.target.eq(1)]
    assert positive.event_episode_id.notna().all()
    event_recall = positive.groupby("event_episode_id")[decision].max().mean()
    return {"AP":average_precision_score(y,frame.raw_score),"ROC-AUC":roc_auc_score(y,frame.raw_score),
        "Precision":precision_score(y,predicted,zero_division=0),"Recall":recall_score(y,predicted),
        "F1":f1_score(y,predicted),"MCC":matthews_corrcoef(y,predicted),
        "Balanced accuracy":balanced_accuracy_score(y,predicted),"Brier":brier_score_loss(y,p),
        "Log loss":log_loss(y,p,labels=[0,1]),"Event Recall":event_recall,
        "False alarms/1000":fp/max(fp+tn,1)*1000,"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),
        "Positive episodes":positive.event_episode_id.nunique(),"n":len(frame)}


metric_rows = []
for (family,scope,seed),frame in development.groupby(["model","scope","seed"],sort=False):
    metric_rows.append({"Model":family,"Scope":scope,"Seed":seed,**evaluate_frame(frame)})
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(TABLES / "nb04_all_metrics_by_seed.csv",index=False)
for scope in ["temporal","transfer"]:
    summary = metrics.query("Scope == @scope").groupby("Model",sort=False).agg(
        **{"Mean AP":("AP","mean"),"SD AP":("AP","std"),"Minimum AP":("AP","min"),
           "Mean ROC-AUC":("ROC-AUC","mean"),"Repeats":("Seed","size")}).reset_index()
    summary["SD AP"] = summary["SD AP"].where(summary.Repeats.gt(1),"Deterministic rule; one run")
    scientific_table(summary.sort_values("Mean AP",ascending=False),f"nb04_ranking_{scope}",
                     f"Development ranking: {scope}; the mean across seeds is not an ensemble prediction")
    representative = metrics.query("Scope == @scope and Seed == 42")
    scientific_table(representative[["Model","Precision","Recall","F1","MCC","Balanced accuracy"]],
        f"nb04_threshold_metrics_{scope}",f"SCRAM classification at the OOF threshold: {scope}, seed 42")
    scientific_table(representative[["Model","Brier","Log loss","Event Recall","False alarms/1000"]],
        f"nb04_probability_event_metrics_{scope}",f"Calibration and event-level diagnostics: {scope}, seed 42")
    scientific_table(representative[["Model","TN","FP","FN","TP"]],
        f"nb04_confusion_{scope}",f"Confusion counts: {scope}, seed 42")


**Development ranking: temporal; the mean across seeds is not an ensemble prediction**

Model,Mean AP,SD AP,Minimum AP,Mean ROC-AUC,Repeats
TCN + context,0.9518,0.0050,0.9463,0.9999,3
Inception + context,0.9263,0.0281,0.8979,0.9998,3
TabM,0.9163,0.0128,0.9035,0.9999,3
MultiRocket + context,0.8900,0.0339,0.8583,0.9998,3
Hydra + context,0.7358,0.0536,0.6776,0.9996,3


**SCRAM classification at the OOF threshold: temporal, seed 42**

Model,Precision,Recall,F1,MCC,Balanced accuracy
MultiRocket + context,0.8125,0.9286,0.8667,0.8683,0.9641
Hydra + context,0.7143,0.8333,0.7692,0.7710,0.9163
Inception + context,0.7273,0.9524,0.8247,0.8319,0.9758
TCN + context,0.7843,0.9524,0.8602,0.8640,0.9759
TabM,0.8696,0.9524,0.9091,0.9098,0.9760


**Calibration and event-level diagnostics: temporal, seed 42**

Model,Brier,Log loss,Event Recall,False alarms/1000
MultiRocket + context,0.0004,0.0019,0.9286,0.4435
Hydra + context,0.0008,0.0029,0.8333,0.6900
Inception + context,0.0003,0.0014,0.9524,0.7392
TCN + context,0.0003,0.0018,0.9524,0.5421
TabM,0.0003,0.0018,0.9524,0.2957


... (6 outputs left out)

**Calibration and event-level diagnostics: transfer, seed 42**

Model,Brier,Log loss,Event Recall,False alarms/1000
MultiRocket + context,0.0007,0.0029,0.8667,0.3559
Hydra + context,0.0009,0.0035,0.6667,0.3559
Inception + context,0.0006,0.0018,0.8000,0.3559
TCN + context,0.0004,0.0015,1.0000,0.3559
TabM,0.0009,0.0034,0.7333,0.3559


**Confusion counts: transfer, seed 42**

Model,TN,FP,FN,TP
MultiRocket + context,5617,2,2,13
Hydra + context,5617,2,5,10
Inception + context,5617,2,3,12
TCN + context,5617,2,0,15
TabM,5617,2,4,11


In [37]:
budget_rows = []
for (family,scope),frame in development.query("seed == 42").groupby(["model","scope"],sort=False):
    m = evaluate_frame(frame,"decision_budget")
    budget_rows.append({"Model":family,"Scope":scope,**{k:m[k] for k in ["Precision","Recall","F1","Event Recall","False alarms/1000"]}})
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(budget_rows).query("Scope == @scope").drop(columns="Scope"),
        f"nb04_alarm_budget_{scope}",f"OOF alarm-budget threshold: observed performance on {scope}")


**OOF alarm-budget threshold: observed performance on temporal**

Model,Precision,Recall,F1,Event Recall,False alarms/1000
MultiRocket + context,0.8125,0.9286,0.8667,0.9286,0.4435
Hydra + context,0.6774,1.0000,0.8077,1.0000,0.9857
Inception + context,0.5467,0.9762,0.7009,0.9762,1.6756
TCN + context,0.5122,1.0000,0.6774,1.0000,1.9713
TabM,0.6562,1.0000,0.7925,1.0000,1.0842


**OOF alarm-budget threshold: observed performance on transfer**

Model,Precision,Recall,F1,Event Recall,False alarms/1000
MultiRocket + context,0.8235,0.9333,0.8750,0.9333,0.5339
Hydra + context,0.8750,0.9333,0.9032,0.9333,0.3559
Inception + context,0.8750,0.9333,0.9032,0.9333,0.3559
TCN + context,0.8333,1.0000,0.9091,1.0000,0.5339
TabM,0.8667,0.8667,0.8667,0.8667,0.3559


## 11. Errors and documented controls

In [38]:
diagnostic = development.query("seed == 42").copy()
row_indices = diagnostic.row_index.to_numpy(int)
diagnostic["power_pct"] = features.iloc[row_indices].power_pct.to_numpy()
diagnostic["delta_1d"] = features.iloc[row_indices].delta_1d.to_numpy()
diagnostic["weather_missing_fraction"] = features.iloc[row_indices].weather_missing_fraction.to_numpy()
diagnostic["change_group"] = pd.cut(np.abs(diagnostic.delta_1d),[-np.inf,2,20,np.inf],
    labels=["Change ≤2 pp","Change >2–20 pp","Change >20 pp"],right=True).astype(str)
diagnostic["weather_group"] = np.where(diagnostic.weather_missing_fraction>0,"At least one weather value missing","Weather complete")
diagnostic["power_group"] = np.where(diagnostic.power_pct.eq(0),"Zero output","Positive output")
hard_rows, subtype_rows, regime_rows = [], [], []
for (family,scope),frame in diagnostic.groupby(["model","scope"],sort=False):
    hard = frame.loc[frame.target.eq(1)|frame.control_kind.eq("Documented non-SCRAM")]
    if hard.target.nunique()==2:
        hm = evaluate_frame(hard)
        hard_rows.append({"Model":family,"Scope":scope,"n":len(hard),"SCRAM prevalence":hard.target.mean(),
            "AP":hm["AP"],"Recall":hm["Recall"],"False alarms/1000":hm["False alarms/1000"]})
    for subtype,part in frame.loc[frame.target.eq(1)].groupby("scram_type",dropna=False):
        subtype_rows.append({"Model":family,"Scope":scope,"SCRAM type":str(subtype),"Positive days":len(part),
            "Detected days":int(part.decision_f1.sum()),"Recall":part.decision_f1.mean()})
    for grouping in ["change_group","power_group","weather_group"]:
        for group,part in frame.groupby(grouping,observed=True):
            positives = part.target.eq(1)
            negatives = part.target.eq(0)
            regime_rows.append({"Model":family,"Scope":scope,"Stratum":group,"Positive days":int(positives.sum()),
                "Negative days":int(negatives.sum()),"Recall":part.loc[positives,"decision_f1"].mean() if positives.any() else "No SCRAM days",
                "False alarms/1000":part.loc[negatives,"decision_f1"].mean()*1000 if negatives.any() else "No control days"})
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(hard_rows).query("Scope == @scope").drop(columns="Scope"),
        f"nb04_hard_controls_{scope}",f"Documented non-SCRAM controls only: {scope}")
    for subtype,part in pd.DataFrame(subtype_rows).query("Scope == @scope").groupby("SCRAM type"):
        scientific_table(part.drop(columns=["Scope","SCRAM type"]),f"nb04_subtype_{subtype}_{scope}",
                         f"Binary detection of rare subtype {subtype}: {scope}")
for (scope,stratum),part in pd.DataFrame(regime_rows).groupby(["Scope","Stratum"],sort=False):
    slug = hashlib.sha256(stratum.encode()).hexdigest()[:8]
    scientific_table(part.drop(columns=["Scope","Stratum"]),f"nb04_regime_{slug}_{scope}",
                     f"{scope}: {stratum}")


**Documented non-SCRAM controls only: temporal**

Model,n,SCRAM prevalence,AP,Recall,False alarms/1000
MultiRocket + context,183,0.2295,0.8699,0.9286,63.8298
Hydra + context,183,0.2295,0.6776,0.8333,99.2908
Inception + context,183,0.2295,0.9673,0.9524,56.7376
TCN + context,183,0.2295,0.9554,0.9524,56.7376
TabM,183,0.2295,0.9045,0.9524,42.5532


**Binary detection of rare subtype A: temporal**

Model,Positive days,Detected days,Recall
MultiRocket + context,30,29,0.9667
Hydra + context,30,26,0.8667
Inception + context,30,29,0.9667
TCN + context,30,29,0.9667
TabM,30,29,0.9667


**Binary detection of rare subtype M: temporal**

Model,Positive days,Detected days,Recall
MultiRocket + context,12,10,0.8333
Hydra + context,12,9,0.7500
Inception + context,12,11,0.9167
TCN + context,12,11,0.9167
TabM,12,11,0.9167


... (2 outputs left out)

**Binary detection of rare subtype A: transfer**

Model,Positive days,Detected days,Recall
MultiRocket + context,10,9,0.9000
Hydra + context,10,7,0.7000
Inception + context,10,8,0.8000
TCN + context,10,10,1.0000
TabM,10,8,0.8000


**Binary detection of rare subtype M: transfer**

Model,Positive days,Detected days,Recall
MultiRocket + context,5,4,0.8000
Hydra + context,5,3,0.6000
Inception + context,5,4,0.8000
TCN + context,5,5,1.0000
TabM,5,3,0.6000


In [39]:
pair_rows = []
for scope in ["temporal","transfer"]:
    frame = diagnostic.query("scope == @scope and model != 'Majority'")
    truth = frame.groupby("observation_id").target.first()
    decisions = frame.pivot(index="observation_id",columns="model",values="decision_f1").loc[truth.index]
    assert decisions.notna().all().all()
    for first,second in itertools.combinations(decisions.columns,2):
        pos = truth.eq(1)
        a,b = decisions.loc[pos,first].astype(bool),decisions.loc[pos,second].astype(bool)
        union_miss = ((~a)|(~b)).sum()
        pair_rows.append({"Scope":scope,"Model A":first,"Model B":second,
            "Only A detects":int((a&~b).sum()),"Only B detects":int((~a&b).sum()),
            "Both miss":int((~a&~b).sum()),
            "Miss Jaccard":float((~a&~b).sum()/union_miss) if union_miss else 0.0})
for scope in ["temporal","transfer"]:
    pair_table = pd.DataFrame(pair_rows).query("Scope == @scope").drop(columns="Scope")
    for block,offset in enumerate(range(0,len(pair_table),5),start=1):
        scientific_table(pair_table.iloc[offset:offset+5],f"nb04_error_overlap_{scope}_{block}",
                         f"Complementarity of missed events: {scope}")


**Complementarity of missed events: temporal**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
Hydra + context,Inception + context,0,5,2,0.2857
Hydra + context,MultiRocket + context,0,4,3,0.4286
Hydra + context,TCN + context,0,5,2,0.2857
Hydra + context,TabM,0,5,2,0.2857
Inception + context,MultiRocket + context,1,0,2,0.6667


**Complementarity of missed events: temporal**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
Inception + context,TCN + context,0,0,2,1.0000
Inception + context,TabM,0,0,2,1.0000
MultiRocket + context,TCN + context,0,1,2,0.6667
MultiRocket + context,TabM,0,1,2,0.6667
TCN + context,TabM,0,0,2,1.0000


**Complementarity of missed events: transfer**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
Hydra + context,Inception + context,1,3,2,0.3333
Hydra + context,MultiRocket + context,0,3,2,0.4000
Hydra + context,TCN + context,0,5,0,0.0000
Hydra + context,TabM,0,1,4,0.8000
Inception + context,MultiRocket + context,0,1,2,0.6667


**Complementarity of missed events: transfer**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
Inception + context,TCN + context,0,3,0,0.0000
Inception + context,TabM,2,1,2,0.4000
MultiRocket + context,TCN + context,0,2,0,0.0000
MultiRocket + context,TabM,2,0,2,0.5000
TCN + context,TabM,4,0,0,0.0000


In [40]:
# Seed 42 and model ordering are fixed; selection of the displayed leader uses development only.
leader = metrics.query("Scope == 'temporal' and Model in @FAMILIES").groupby("Model").AP.mean().idxmax()
leader_frame = diagnostic.query("scope == 'temporal' and model == @leader")
site_rows, reactor_rows = [], []
for grouping,container in [("plant_id",site_rows),("reactor_id",reactor_rows)]:
    for name,part in leader_frame.groupby(grouping,sort=True):
        positive,negative = part.target.eq(1),part.target.eq(0)
        container.append({"Site" if grouping=="plant_id" else "Reactor":name,
            "Positive days":int(positive.sum()),"Negative days":int(negative.sum()),
            "TP":int((positive&part.decision_f1).sum()),"FN":int((positive&~part.decision_f1).sum()),
            "FP":int((negative&part.decision_f1).sum()),
            "Recall":part.loc[positive,"decision_f1"].mean() if positive.any() else "No SCRAM days"})
for label,rows in [("site",site_rows),("reactor",reactor_rows)]:
    table = pd.DataFrame(rows)
    table.to_csv(TABLES / f"nb04_all_{label}_errors.csv",index=False)
    # Retain every site/reactor with an event or a false alarm in the CSV ledgers; the narrative keeps missed-event evidence.
    informative = table.loc[table["Positive days"].gt(0)|table.FP.gt(0)].copy()
    for block,offset in enumerate(range(0,len(informative),8),start=1):
        scientific_table(informative.iloc[offset:offset+8],f"nb04_{label}_errors_{block}",
                         f"{leader}: {label}, events or false alarms; all eligible groups")
all_errors = diagnostic.loc[diagnostic.target.ne(diagnostic.decision_f1.astype(int))].copy()
all_errors.to_parquet(PRED_ROOT / "development_error_ledger.parquet",index=False)
misses = diagnostic.query("scope == 'temporal' and model != 'Majority' and target == 1 and decision_f1 == False")
missed_events = misses.groupby(["observation_id","date","reactor_id","event_id"],dropna=False).agg(
    **{"Missed by":("model",lambda s:", ".join(sorted(set(s)))),"Power (%)":("power_pct","first"),
       "Change (pp)":("delta_1d","first"),"Weather missing fraction":("weather_missing_fraction","first")}).reset_index().drop(columns="observation_id")
for block,offset in enumerate(range(0,len(missed_events),6),start=1):
    scientific_table(missed_events.iloc[offset:offset+6].fillna({"event_id":"Not specified by source"}),f"nb04_missed_events_{block}",
                     f"All positive days missed by at least one substantive comparator")


**All positive days missed by at least one substantive comparator**

date,reactor_id,event_id,Missed by,Power (%),Change (pp),Weather missing fraction
2023-08-12 00:00:00,05000265,56676,Hydra + context,0.0000,−100.0000,0.0000
2023-11-15 00:00:00,05000364,56852,"Hydra + context, Inception + context, MultiRocket + context, TCN + context, TabM",0.0000,−1.0000,0.0000
2023-11-18 00:00:00,05000458,56863,Hydra + context,0.0000,−21.0000,0.0000
2023-12-14 00:00:00,05000458,56887,Hydra + context,0.0000,0.0000,0.0000
2023-03-16 00:00:00,05200025,56414,Hydra + context,0.0000,−18.0000,0.0000
2023-04-11 00:00:00,05200025,56460,"Hydra + context, Inception + context, MultiRocket + context, TCN + context, TabM",0.0000,0.0000,0.0000


**All positive days missed by at least one substantive comparator**

date,reactor_id,event_id,Missed by,Power (%),Change (pp),Weather missing fraction
2023-05-03 00:00:00,05200025,56497,"Hydra + context, MultiRocket + context",0.0000,0.0000,0.0000


In [41]:
pr_rows = []
for (family,scope),part in diagnostic.groupby(["model","scope"],sort=False):
    precision,recall,threshold = precision_recall_curve(part.target,part.raw_score)
    row = {"Model":family,"Scope":scope}
    for goal in [0.5,0.75,0.9,1.0]:
        row[f"Precision at recall ≥{goal:.2f}"] = float(precision[recall>=goal-1e-12].max())
    pr_rows.append(row)
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(pr_rows).query("Scope == @scope").drop(columns="Scope"),
        f"nb04_pr_profile_{scope}",f"Descriptive development PR points: {scope}")
reliability = leader_frame[["probability","target"]].copy()
reliability["bin"] = pd.qcut(reliability.probability,10,duplicates="drop")
reliability_table = reliability.groupby("bin",observed=True).agg(
    n=("target","size"),positive_days=("target","sum"),mean_probability=("probability","mean"),
    observed_frequency=("target","mean"),minimum_probability=("probability","min"),maximum_probability=("probability","max")).reset_index(drop=True)
reliability_table.insert(0,"Bin",np.arange(1,len(reliability_table)+1))
reliability_table["positive_days"] = reliability_table.positive_days.astype(int)
scientific_table(reliability_table,"nb04_reliability_leader",f"Probability reliability: {leader}, seed 42")


**Descriptive development PR points: temporal**

Model,Precision at recall ≥0.50,Precision at recall ≥0.75,Precision at recall ≥0.90,Precision at recall ≥1.00
MultiRocket + context,0.9024,0.9024,0.8837,0.3134
Hydra + context,0.7692,0.7619,0.7273,0.7119
Inception + context,0.9744,0.9744,0.9744,0.5250
TCN + context,0.9600,0.9487,0.9268,0.7778
TabM,0.9412,0.9412,0.8889,0.7368


**Descriptive development PR points: transfer**

Model,Precision at recall ≥0.50,Precision at recall ≥0.75,Precision at recall ≥0.90,Precision at recall ≥1.00
MultiRocket + context,0.8889,0.8667,0.8235,0.7500
Hydra + context,0.9091,0.8824,0.8824,0.8824
Inception + context,1.0000,0.8824,0.8824,0.8824
TCN + context,0.9231,0.9231,0.8824,0.8824
TabM,0.9167,0.8824,0.8824,0.8824


**Probability reliability: TCN + context, seed 42**

Bin,n,positive_days,mean_probability,observed_frequency,minimum_probability,maximum_probability
1,2034,0,1.12e−05,0.0000,5.02e−07,1.75e−05
2,2033,0,2.18e−05,0.0000,1.75e−05,2.60e−05
3,2033,0,3.05e−05,0.0000,2.60e−05,3.49e−05
4,2033,0,3.86e−05,0.0000,3.49e−05,4.26e−05
5,2034,0,4.88e−05,0.0000,4.26e−05,5.64e−05
6,2033,0,6.60e−05,0.0000,5.64e−05,7.67e−05
7,2033,0,9.65e−05,0.0000,7.67e−05,0.0001
8,2033,0,0.0001,0.0000,0.0001,0.0002
9,2033,0,0.0002,0.0000,0.0002,0.0003
10,2034,42,0.0285,0.0206,0.0003,0.9995


## 12. Cost of temporal representations

In [43]:
resource_frame=pd.DataFrame(resource_rows)
for scope in ["temporal","transfer"]:
    final=resource_frame.query("scope == @scope and fold == 'development'").groupby("model",sort=False).agg(
        **{"Mean fit (s)":("fit_seconds","mean"),"Mean full inference (s)":("batch_predict_seconds","mean"),"Parameters":("parameters","first"),"Checkpoint MiB":("model_bytes",lambda s:s.mean()/2**20),"Peak process RSS MiB":("Peak process RSS MiB","max")}).reset_index()
    scientific_table(final,"nb04_efficiency_"+scope,f"Computational cost: {scope}")
curves=pd.DataFrame(learning_rows)
for family in ["inception","tcn","tabm"]:
    parts=[]
    for scope in ["temporal","transfer"]:
        selected_runs=[f"{family}_{scope}_development_c{chosen[(name,scope)]}_s42" for name,slug in model_slugs.items() if slug==family]
        parts.append(curves.loc[curves.Run.isin(selected_runs)].assign(Scope=scope))
    learning_table=pd.concat(parts).pivot(index="Epoch",columns="Scope",values="Training BCE").reset_index()
    scientific_table(learning_table,"nb04_learning_"+family,f"Training BCE: {family}, seed 42; losses under different class ratios are not cross-protocol performance metrics")
selected=[{"model":family,"scope":scope,"configuration":config_id,"parameters":MODERN_GRIDS[family][config_id-1]} for (family,scope),config_id in chosen.items()]
(OUTPUT_ROOT / "Configs/selected_modern.json").write_text(json.dumps(selected,indent=2));


**Computational cost: temporal**

model,Mean fit (s),Mean full inference (s),Parameters,Checkpoint MiB,Peak process RSS MiB
MultiRocket + context,199.4829,2.9993,4098,0.0984,5931.6562
Hydra + context,192.4167,26.0871,2114,0.0684,4096.8125
Inception + context,202.8744,0.3316,15521,0.0633,1826.0312
TCN + context,83.0112,0.2245,21601,0.0813,1705.8906
TabM,137.6085,0.4779,223552,0.7874,1457.5312


**Computational cost: transfer**

model,Mean fit (s),Mean full inference (s),Parameters,Checkpoint MiB,Peak process RSS MiB
MultiRocket + context,54.9469,0.3100,2082,0.0514,3315.7656
Hydra + context,133.6903,7.4969,2114,0.0684,2944.9375
Inception + context,205.8821,0.1696,15521,0.0634,1537.2656
TCN + context,57.0345,0.0820,21601,0.0812,1442.2031
TabM,58.0772,0.4094,80064,0.2843,1179.5000


**Training BCE: inception, seed 42; losses under different class ratios are not cross-protocol performance metrics**

Epoch,temporal,transfer
1,0.0373,0.0442
2,0.0164,0.0139
3,0.0160,0.0132
4,0.0133,0.0109
5,0.0129,0.0067
6,0.0119,0.0078
7,0.0107,0.0061
8,0.0115,0.0061
9,0.0092,0.0050
10,0.0082,0.0056


**Training BCE: tcn, seed 42; losses under different class ratios are not cross-protocol performance metrics**

Epoch,temporal,transfer
1,0.0540,0.0509
2,0.0167,0.0173
3,0.0183,0.0161
4,0.0161,0.0150
5,0.0158,0.0111
6,0.0140,0.0119
7,0.0129,0.0091
8,0.0142,0.0098
9,0.0138,0.0077
10,0.0139,0.0074


**Training BCE: tabm, seed 42; losses under different class ratios are not cross-protocol performance metrics**

Epoch,temporal,transfer
1,0.0569,0.0585
2,0.0207,0.0208
3,0.0193,0.0203
4,0.0173,0.0180
5,0.0170,0.0142
6,0.0166,0.0145
7,0.0149,0.0137
8,0.0163,0.0139
9,0.0147,0.0125
10,0.0146,0.0119


## 13. Baseline comparison and implications for RACF-Net

In [44]:
baseline_metrics=pd.read_csv(TABLES / "nb03_all_metrics_by_seed.csv")
combined=pd.concat([baseline_metrics,metrics],ignore_index=True)
for scope in ["temporal","transfer"]:
    comparison=combined.query("Scope == @scope").groupby("Model",sort=False).agg(**{"Mean AP":("AP","mean"),"Minimum AP":("AP","min"),"Mean F1":("F1","mean"),"Mean Brier":("Brier","mean")}).reset_index().sort_values("Mean AP",ascending=False)
    scientific_table(comparison,"nb04_all_families_"+scope,f"Baselines and modern model families: {scope}")
baseline_development=pd.read_parquet(OUTPUT_ROOT / "Predictions" / RUN_ID / "baselines/development_predictions.parquet")
all_development=pd.concat([baseline_development,development],ignore_index=True)
all_development.to_parquet(PRED_ROOT / "all_competitors_development.parquet",index=False)
(OUTPUT_ROOT / "Configs/modern_design_evidence.json").write_text(json.dumps({"temporal_mean_AP":combined.query("Scope == 'temporal'").groupby("Model").AP.mean().to_dict(),"transfer_mean_AP":combined.query("Scope == 'transfer'").groupby("Model").AP.mean().to_dict(),"test_used":False},indent=2));


**Baselines and modern model families: temporal**

Model,Mean AP,Minimum AP,Mean F1,Mean Brier
TCN + context,0.9518,0.9463,0.8629,0.0004
LightGBM,0.9480,0.9480,0.9176,0.0003
CatBoost,0.9405,0.9321,0.9291,0.0003
Logistic Regression,0.9340,0.9340,0.8500,0.0004
Inception + context,0.9263,0.8979,0.8428,0.0006
TabM,0.9163,0.9035,0.8987,0.0004
Recent-level deviation,0.9135,0.9135,0.8193,0.0007
Random Forest,0.9045,0.9021,0.9154,0.0005
MultiRocket + context,0.8900,0.8583,0.8726,0.0004
Power-drop rule,0.7690,0.7690,0.8049,0.0008


**Baselines and modern model families: transfer**

Model,Mean AP,Minimum AP,Mean F1,Mean Brier
TCN + context,0.9212,0.9087,0.8600,0.0006
Logistic Regression,0.9041,0.9041,0.7692,0.0009
Inception + context,0.9022,0.8134,0.7875,0.0009
TabM,0.9008,0.8808,0.8519,0.0008
CatBoost,0.8835,0.8708,0.8536,0.0006
Recent-level deviation,0.8782,0.8782,0.6923,0.0011
MultiRocket + context,0.8702,0.8515,0.8397,0.0008
Hydra + context,0.8676,0.8205,0.7847,0.0009
LightGBM,0.8435,0.8435,0.8667,0.0007
Random Forest,0.8059,0.7766,0.8276,0.0008


## 14. Checkpoint and solver checks

In [45]:
def predict_package(package,indices):
    if package["family"]=="TabM":
        z=np.clip(package["representation"].table.transform(features.iloc[indices]),-20,20).astype(np.float32)
        model=tabm_network(package["configuration"],package["context_dim"]);model.load_state_dict(package["state_dict"])
        scores=tabm_scores(model,z);del model
        return scores
    x,z=package["representation"].transform(indices)
    if "transform" in package:
        design=np.concatenate([package["scale"].transform(convolution_features(package["transform"],x)),z],axis=1)
        return 1/(1+np.exp(-np.clip(package["classifier"].decision_function(design),-25,25)))
    model=TemporalComparator(package["family"],package["channels"],package["context_dim"],package["configuration"]["width"]).to(DEVICE)
    model.load_state_dict(package["state_dict"])
    p=neural_scores(model,x,z)
    del model
    if DEVICE.type=="mps":torch.mps.empty_cache()
    return p

reload_rows=[]
for family in FAMILIES:
    for scope in ["temporal","transfer"]:
        package=joblib.load(MODEL_ROOT / f"{model_slugs[family]}_{scope}_seed42.joblib")
        reference=development.query("model == @family and scope == @scope and seed == 42")
        p=predict_package(package,reference.row_index.to_numpy(int))
        difference=float(np.max(np.abs(p-reference.raw_score.to_numpy())))
        assert difference<=1e-6
        reload_rows.append({"Model":family,"Scope":scope,"n":len(p),"Maximum absolute difference":difference})
        del package;gc.collect()
scientific_table(pd.DataFrame(reload_rows),"nb04_checkpoint_reproduction","Reproducibility after checkpoint reload")

solver_rows=[]
for family in ["MultiRocket + context","Hydra + context"]:
    for scope in ["temporal","transfer"]:
        for seed in SEEDS:
            package=joblib.load(MODEL_ROOT / f"{model_slugs[family]}_{scope}_seed{seed}.joblib")
            classifier=package["classifier"];iterations=int(np.max(classifier.n_iter_))
            assert iterations<classifier.max_iter,"Selected linear head reached the iteration limit; repair before freeze"
            solver_rows.append({"Model":family,"Scope":scope,"Seed":seed,"LSQR iterations":iterations,"Iteration limit":classifier.max_iter,"Limit reached":False})
            del package
scientific_table(pd.DataFrame(solver_rows),"nb04_selected_solver_convergence","Selected linear-head convergence: the iteration limit must not be reached; search-fit iteration counts were not retained separately")
